In [ ]:
import os
import glob

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import rpy2.robjects as ro
from rpy2.robjects import FloatVector, r

In [ ]:
#INSTALLING PACKAGES

In [ ]:
# Folder where we install R packages
local_r_lib = os.path.expanduser("~/Rlibs")

# Folder with local .tar.gz archives
r_packages_dir = os.path.abspath("R_packages")

print("Current working directory:")
print(os.getcwd())

print("\nR packages directory:")
print(r_packages_dir)

if not os.path.isdir(r_packages_dir):
    raise FileNotFoundError(f"Папка R_packages не найдена:\n{r_packages_dir}")

# IMPORTANT: Installation procedure
packages_order = [
    "SparseM",
    "quadprog",
    "lpSolve",
    "limSolve",
    "baseline",
]

package_files = {}

for pkg in packages_order:
    matches = sorted(glob.glob(os.path.join(r_packages_dir, f"{pkg}_*.tar.gz")))
    
    if len(matches) == 0:
        raise FileNotFoundError(
            f"\nArchive for package {pkg} not found in R_packages folder.\n"
            f"A file of the type was expected: {pkg}_*.tar.gz\n\n"
            f"Now in the R_packages folder there are:\n"
            + "\n".join(os.listdir(r_packages_dir))
        )
    
    package_files[pkg] = matches[-1]

print("\nFound package archives:")
for pkg, path in package_files.items():
    print(f"{pkg}: {path}")

# Passing paths to R
ro.globalenv["local_r_lib"] = local_r_lib
ro.globalenv["package_names"] = ro.StrVector(packages_order)
ro.globalenv["package_paths"] = ro.StrVector([package_files[pkg] for pkg in packages_order])

r_code = r'''
dir.create(local_r_lib, showWarnings = FALSE, recursive = TRUE)
.libPaths(c(local_r_lib, .libPaths()))

cat("\nR version:\n")
print(R.version.string)

cat("\nR library paths:\n")
print(.libPaths())

# Delete lock files from failed installations
lock_dirs <- list.files(local_r_lib, pattern = "^00LOCK", full.names = TRUE)
if (length(lock_dirs) > 0) {
    cat("\nRemoving old lock directories:\n")
    print(lock_dirs)
    unlink(lock_dirs, recursive = TRUE, force = TRUE)
}

install_local_package <- function(pkg_name, pkg_file) {
    cat("\n----------------------------------------\n")
    cat("Installing package:", pkg_name, "\n")
    cat("From file:", pkg_file, "\n")
    
    if (!file.exists(pkg_file)) {
        stop(paste("File not found:", pkg_file))
    }
    
    # Remove the old/broken installation of this package
    pkg_dir <- file.path(local_r_lib, pkg_name)
    if (dir.exists(pkg_dir)) {
        cat("Removing existing package directory:", pkg_dir, "\n")
        unlink(pkg_dir, recursive = TRUE, force = TRUE)
    }
    
    install.packages(
        pkg_file,
        lib = local_r_lib,
        repos = NULL,
        type = "source",
        INSTALL_opts = c("--no-lock")
    )
    
    if (!requireNamespace(pkg_name, quietly = TRUE, lib.loc = local_r_lib)) {
        stop(paste("Package was not installed correctly:", pkg_name))
    }
    
    cat("Successfully installed:", pkg_name, "\n")
}

# Install all packages in the correct order
for (i in seq_along(package_names)) {
    install_local_package(package_names[[i]], package_paths[[i]])
}

cat("\n----------------------------------------\n")
cat("Trying to load baseline...\n")

library(baseline, lib.loc = local_r_lib)

cat("\nPackage baseline installed and loaded successfully!\n")
'''

ro.r(r_code)

In [ ]:
#LOADING PACKAGES

In [ ]:
ro.r('dir.create("~/Rlibs", showWarnings = FALSE, recursive = TRUE)')
ro.r('.libPaths(c("~/Rlibs", .libPaths()))')
ro.r('library(baseline)')

print("Package baseline loaded successfully!")

In [ ]:
#MAIN

In [ ]:
red_db_red = pd.read_csv("../2_compiling_unified_database/MICROSCAN_database.csv")

In [ ]:
red_db_red.head()

In [ ]:
red_db_red_left = red_db_red[['File name', 'Color', 'Polymer', 'Matching']]
red_db_red_right = red_db_red.drop(['File name', 'Color', 'Polymer', 'Matching'], axis=1)

spectral_columns = list(red_db_red_right.columns)
x = np.array([float(i) for i in spectral_columns]) 

# Process each row in the DataFrame
fouling_indices = []
for idx, row in tqdm(red_db_red_right.iterrows()):
    y = np.array(list(row))     # Spectral data

    r_spectral_data = FloatVector(y)
    r_matrix = r['matrix'](r_spectral_data, nrow=1, ncol=len(r_spectral_data))
    fourS = ro.r('baseline.fillPeaks')
    fourS_result = fourS(r_matrix, 4, 25, 10, 355)
    corrected_data_4s = [max(0, x) for x in list(fourS_result[1])] #list(fourS_result[1])
    baseline_data_4s = list(fourS_result[0])
    
    yy_reversed = corrected_data_4s
    assert not np.isnan(yy_reversed).any(), f"specy_prep produced NaN: {yy_reversed}"

    red_db_red_right.loc[idx] = corrected_data_4s

In [ ]:
red_db_red_baseline = pd.concat([red_db_red_left, red_db_red_right], axis=1)
red_db_red_baseline.head()

In [ ]:
red_db_red_baseline.to_csv('MICROSCAN_database_baseline_corrected.csv', index=False)